In [2]:
import sys, os, time
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt

# Konfigurasi Path
sys.path.append('/content/drive/MyDrive/simuteh-2330511031/src')
os.chdir('/content/drive/MyDrive/simuteh-2330511031/notebook')
from utils_cv import baca, tampil, kontras_bercak

# --- Siapkan Data ---
nama = "teh_041.png"
bersih = cv2.cvtColor(baca(f"../data/sintetis/bersih/{nama}"), cv2.COLOR_BGR2GRAY)
md = cv2.imread(f"../data/sintetis/mask_daun/{nama}", 0)
mb = cv2.imread(f"../data/sintetis/mask_bercak/{nama}", 0)
rng = np.random.default_rng(42)

# ---------------- L4.2: Konvolusi Manual vs Pustaka ----------------
print("--- L4.2: Verifikasi Konvolusi ---")
def konvolusi(img, k):
    kh, kw = k.shape; ph, pw = kh // 2, kw // 2
    pad = np.pad(img.astype(float), ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros(img.shape)
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            out[y, x] = np.sum(pad[y:y+kh, x:x+kw] * k)
    return np.clip(np.round(out), 0, 255).astype(np.uint8)

def konvolusi_sejati(img, k):
    return konvolusi(img, np.flip(k))

for nm, k in {"rerata": np.ones((3, 3)) / 9, "asimetris": np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], float)}.items():
    t0 = time.perf_counter(); a = konvolusi(bersih, k); t_manual = time.perf_counter() - t0
    t0 = time.perf_counter(); b = cv2.filter2D(bersih, -1, k, borderType=cv2.BORDER_REPLICATE); t_cv = time.perf_counter() - t0
    beda = np.abs(a.astype(int) - b.astype(int)).max()
    c = konvolusi_sejati(bersih, k); beda2 = np.abs(c.astype(int) - b.astype(int)).max()
    print(f"{nm:10s}: selisih manual vs filter2D={beda}, sejati vs filter2D={beda2}, waktu manual={t_manual:.3f} s, filter2D={t_cv:.5f} s")

# ---------------- L4.3: Tiga Penapis, Dua Jenis Derau ----------------
print("\n--- L4.3: Perbandingan Penapis (Membuat Tabel) ---")
def derau_gauss(img, sd): return np.clip(img + rng.normal(0, sd, img.shape), 0, 255).astype(np.uint8)
def derau_sp(img, p):
    out = img.copy(); m = rng.random(img.shape)
    out[m < p / 2] = 0; out[m > 1 - p / 2] = 255
    return out
def ukur_waktu(f, x, ulang=20):
    t = []
    for _ in range(ulang): t0 = time.perf_counter(); f(x); t.append(time.perf_counter() - t0)
    return 1000 * np.mean(t)

derau = {"gauss_10": derau_gauss(bersih, 10), "gauss_20": derau_gauss(bersih, 20), "gauss_40": derau_gauss(bersih, 40), "sp_4%": derau_sp(bersih, 0.04), "sp_10%": derau_sp(bersih, 0.10)}
penapis = {"rerata_3": lambda x: cv2.blur(x, (3, 3)), "rerata_5": lambda x: cv2.blur(x, (5, 5)), "gauss_5": lambda x: cv2.GaussianBlur(x, (5, 5), 1.0), "median_3": lambda x: cv2.medianBlur(x, 3), "median_5": lambda x: cv2.medianBlur(x, 5), "median_9": lambda x: cv2.medianBlur(x, 9)}

baris = []
for dn, d in derau.items():
    baris.append([dn, "tanpa", cv2.PSNR(bersih, d), 0.0, kontras_bercak(d, md, mb)])
    for pn, f in penapis.items():
        out = f(d)
        baris.append([dn, pn, cv2.PSNR(bersih, out), ukur_waktu(f, d), kontras_bercak(out, md, mb)])

tabel = pd.DataFrame(baris, columns=["derau", "penapis", "PSNR_dB", "waktu_ms", "KB"]).round(2)
tabel.to_csv("../hasil/b4_tabel_penapis.csv", index=False)
print("File b4_tabel_penapis.csv berhasil dibuat.")

# ---------------- L4.4: Pengaruh Kernel pada Bercak Kecil ----------------
print("\n--- L4.4: Evaluasi Ukuran Kernel pada Kelas Ringan ---")
def selisih_rerata(kanal, mask_daun, mask_bercak):
    k = kanal.astype(float); sehat = (mask_daun > 0) & (mask_bercak == 0)
    return abs(k[mask_bercak > 0].mean() - k[sehat].mean())

label = pd.read_csv("../data/sintetis/label.csv")
baris_ringan = []
for nm in label[label.kelas == "ringan"].nama_file:
    g = cv2.cvtColor(baca(f"../data/sintetis/bersih/{nm}"), cv2.COLOR_BGR2GRAY)
    md2 = cv2.imread(f"../data/sintetis/mask_daun/{nm}", 0)
    mb2 = cv2.imread(f"../data/sintetis/mask_bercak/{nm}", 0)
    for k in [1, 3, 5, 9, 15]:
        out = g if k == 1 else cv2.medianBlur(g, k)
        baris_ringan.append([nm, k, selisih_rerata(out, md2, mb2), kontras_bercak(out, md2, mb2)])

hasil_median = pd.DataFrame(baris_ringan, columns=["nama", "kernel", "dI", "KB"])
print(hasil_median.groupby("kernel")[["dI", "KB"]].mean().round(2))

# ---------------- L4.5: Urutan Meredam dan Menajamkan ----------------
print("\n--- L4.5: Pengaruh Urutan Penajaman ---")
def unsharp(x, k=1.0):
    blur = cv2.GaussianBlur(x, (5, 5), 1.0)
    return cv2.addWeighted(x, 1 + k, blur, -k, 0)

noisy = derau["gauss_20"]
latar = md == 0
for nm, redam in {"Gaussian": lambda x: cv2.GaussianBlur(x, (5, 5), 1.0), "Median": lambda x: cv2.medianBlur(x, 5)}.items():
    A = redam(unsharp(noisy))
    B = unsharp(redam(noisy))
    print(f"{nm:8s}: tajam->redam PSNR={cv2.PSNR(bersih, A):.2f} sigma_latar={A[latar].std():.1f} | redam->tajam PSNR={cv2.PSNR(bersih, B):.2f} sigma_latar={B[latar].std():.1f}")

# ---------------- L4.6: Tugas Mandiri Penapisan Sensor Bawaan ----------------
print("\n--- L4.6: Penapisan Derau Sensor Asli (Citra Normal) ---")
citra_uji = [f"teh_{str(i).zfill(3)}.png" for i in range(1, 61, 5)]
hasil_psnr = []
for nm in citra_uji:
    bsh = cv2.cvtColor(baca(f"../data/sintetis/bersih/{nm}"), cv2.COLOR_BGR2GRAY)
    ctr = cv2.cvtColor(baca(f"../data/sintetis/citra/{nm}"), cv2.COLOR_BGR2GRAY)
    tapis = cv2.GaussianBlur(ctr, (5, 5), 1.0)
    psnr_awal = cv2.PSNR(bsh, ctr)
    psnr_akhir = cv2.PSNR(bsh, tapis)
    hasil_psnr.append([nm, psnr_awal, psnr_akhir])

tabel_sensor = pd.DataFrame(hasil_psnr, columns=["Citra", "PSNR Awal", "PSNR Sesudah (Gauss 5x5)"]).round(2)
print(tabel_sensor)

--- L4.2: Verifikasi Konvolusi ---
rerata    : selisih manual vs filter2D=0, sejati vs filter2D=0, waktu manual=2.302 s, filter2D=0.02061 s
asimetris : selisih manual vs filter2D=0, sejati vs filter2D=255, waktu manual=3.512 s, filter2D=0.00041 s

--- L4.3: Perbandingan Penapis (Membuat Tabel) ---
File b4_tabel_penapis.csv berhasil dibuat.

--- L4.4: Evaluasi Ukuran Kernel pada Kelas Ringan ---
           dI    KB
kernel             
1       24.30  3.82
3       24.57  3.92
5       23.40  3.78
9       16.25  3.03
15       4.58  1.24

--- L4.5: Pengaruh Urutan Penajaman ---
Gaussian: tajam->redam PSNR=30.36 sigma_latar=7.9 | redam->tajam PSNR=30.20 sigma_latar=8.2
Median  : tajam->redam PSNR=30.16 sigma_latar=8.1 | redam->tajam PSNR=31.11 sigma_latar=7.2

--- L4.6: Penapisan Derau Sensor Asli (Citra Normal) ---
          Citra  PSNR Awal  PSNR Sesudah (Gauss 5x5)
0   teh_001.png      32.47                     40.74
1   teh_006.png      30.26                     39.58
2   teh_011.png     

In [3]:
from google.colab import drive
import os
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
